<a href="https://colab.research.google.com/github/Asritha0507/ML-Market-Basket-Analysis/blob/main/04_Temporal_Candidate_Generation_and_Training_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Notebook 4
# Temporal Candidate Generation and Training Dataset Construction

import pandas as pd
import numpy as np
import os
import gc

print("Notebook 4 started successfully.")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Notebook 4 started successfully.
Pandas version: 2.2.3
NumPy version: 2.1.3


In [ ]:
# Step 2: Mount Google Drive and define dataset paths

from google.colab import drive

drive.mount("/content/drive")

base_path = "/content/drive/MyDrive/ML_Market_Basket_Analysis/Datasets"
feature_path = f"{base_path}/Features"

print("Base path:", base_path)
print("Feature path:", feature_path)

print("\nDataset files:")
print(os.listdir(base_path))

print("\nFeature files:")
print(os.listdir(feature_path))

Mounted at /content/drive
Base path: /content/drive/MyDrive/ML_Market_Basket_Analysis/Datasets
Feature path: /content/drive/MyDrive/ML_Market_Basket_Analysis/02_Datasets/Features

Dataset files:
['departments.csv', 'orders.csv', 'products.csv', 'aisles.csv', 'order_products_prior.csv', 'order_products_train.csv', 'Features']

Feature files:
['user_features.parquet', 'product_features.parquet', 'user_basket_features.parquet', 'user_product_features.parquet']


In [ ]:
# Step 3: Load orders data

orders = pd.read_csv(
    f"{base_path}/orders.csv",
    usecols=[
        "order_id",
        "user_id",
        "eval_set",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order"
    ],
    dtype={
        "order_id": "int32",
        "user_id": "int32",
        "eval_set": "category",
        "order_number": "int16",
        "order_dow": "int8",
        "order_hour_of_day": "int8",
        "days_since_prior_order": "float32"
    }
)

print("Orders shape:", orders.shape)
print("\nColumns:")
print(orders.columns.tolist())

print("\nEvaluation set counts:")
print(orders["eval_set"].value_counts())

print("\nMissing values:")
print(orders.isnull().sum())

Orders shape: (3421083, 7)

Columns:
['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']

Evaluation set counts:
eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64

Missing values:
order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64


In [ ]:
# Step 4: Separate prior, train, and test orders

prior_orders = orders[orders["eval_set"] == "prior"].copy()
train_orders = orders[orders["eval_set"] == "train"].copy()
test_orders = orders[orders["eval_set"] == "test"].copy()

print("Prior orders:", prior_orders.shape)
print("Train orders:", train_orders.shape)
print("Test orders:", test_orders.shape)

print("\nOrder number ranges:")
print("Prior:", prior_orders["order_number"].min(), "to", prior_orders["order_number"].max())
print("Train:", train_orders["order_number"].min(), "to", train_orders["order_number"].max())
print("Test:", test_orders["order_number"].min(), "to", test_orders["order_number"].max())

print("\nUnique users:")
print("Prior:", prior_orders["user_id"].nunique())
print("Train:", train_orders["user_id"].nunique())
print("Test:", test_orders["user_id"].nunique())

Prior orders: (3214874, 7)
Train orders: (131209, 7)
Test orders: (75000, 7)

Order number ranges:
Prior: 1 to 99
Train: 4 to 100
Test: 4 to 100

Unique users:
Prior: 206209
Train: 131209
Test: 75000


In [ ]:
# Step 5: Create temporal target orders

MIN_HISTORY_ORDERS = 3

eligible_targets = prior_orders[
    prior_orders["order_number"] > MIN_HISTORY_ORDERS
][
    ["order_id", "user_id", "order_number",
     "order_dow", "order_hour_of_day",
     "days_since_prior_order"]
].copy()

eligible_targets = eligible_targets.sort_values(
    ["user_id", "order_number"]
)

target_count = eligible_targets.groupby("user_id")["order_id"].transform("size")
position = eligible_targets.groupby("user_id").cumcount()

first_target = position == 0
middle_target = position == target_count // 2
last_target = position == target_count - 1

temporal_targets = eligible_targets[
    first_target | middle_target | last_target
].copy()

temporal_targets = temporal_targets.drop_duplicates(
    subset=["user_id", "order_id"]
).reset_index(drop=True)

print("Eligible target orders:", len(eligible_targets))
print("Selected temporal targets:", len(temporal_targets))
print("Unique customers represented:", temporal_targets["user_id"].nunique())

print("\nTargets per customer:")
print(
    temporal_targets.groupby("user_id")
    .size()
    .value_counts()
    .sort_index()
)

print("\nTarget order number statistics:")
print(temporal_targets["order_number"].describe())

Eligible target orders: 2596247
Selected temporal targets: 491324
Unique customers represented: 182223

Targets per customer:
1     19590
2     16165
3    146468
Name: count, dtype: int64

Target order number statistics:
count    491324.000000
mean         11.424817
std          12.732408
min           4.000000
25%           4.000000
50%           6.000000
75%          13.000000
max          99.000000
Name: order_number, dtype: float64


In [ ]:
# Step 6: Load prior order-product interactions

prior_interactions = pd.read_csv(
    f"{base_path}/order_products_prior.csv",
    usecols=["order_id", "product_id"],
    dtype={
        "order_id": "int32",
        "product_id": "int32"
    }
)

print("Prior interactions shape:", prior_interactions.shape)
print("Unique orders:", prior_interactions["order_id"].nunique())
print("Unique products:", prior_interactions["product_id"].nunique())

print("\nMissing values:")
print(prior_interactions.isnull().sum())

print("\nMemory usage:",
      round(prior_interactions.memory_usage(deep=True).sum() / 1024**2, 2),
      "MB")

Prior interactions shape: (32434489, 2)
Unique orders: 3214874
Unique products: 49677

Missing values:
order_id      0
product_id    0
dtype: int64

Memory usage: 247.46 MB


In [ ]:
# Step 7: Add user and order number to prior interactions

historical_interactions = prior_interactions.merge(
    prior_orders[
        ["order_id", "user_id", "order_number"]
    ],
    on="order_id",
    how="left"
)

print("Historical interactions shape:", historical_interactions.shape)

print("\nColumns:")
print(historical_interactions.columns.tolist())

print("\nMissing values:")
print(historical_interactions.isnull().sum())

print("\nSample:")
print(historical_interactions.head())

Historical interactions shape: (32434489, 4)

Columns:
['order_id', 'product_id', 'user_id', 'order_number']

Missing values:
order_id        0
product_id      0
user_id         0
order_number    0
dtype: int64

Sample:
   order_id  product_id  user_id  order_number
0         2       33120   202279             3
1         2       28985   202279             3
2         2        9327   202279             3
3         2       45918   202279             3
4         2       30035   202279             3


In [ ]:
# Step 8: Build user-product purchase history

user_product_history = (
    historical_interactions
    .groupby(["user_id", "product_id"])["order_number"]
    .agg(["min", "max", "count"])
    .reset_index()
)

user_product_history.columns = [
    "user_id",
    "product_id",
    "first_order",
    "last_order",
    "times_purchased"
]

print("User-product history shape:", user_product_history.shape)

print("\nUnique users:",
      user_product_history["user_id"].nunique())

print("Unique products:",
      user_product_history["product_id"].nunique())

print("\nMissing values:")
print(user_product_history.isnull().sum())

print("\nSample:")
print(user_product_history.head())

User-product history shape: (13307953, 5)

Unique users: 206209
Unique products: 49677

Missing values:
user_id            0
product_id         0
first_order        0
last_order         0
times_purchased    0
dtype: int64

Sample:
   user_id  product_id  first_order  last_order  times_purchased
0        1         196            1          10               10
1        1       10258            2          10                9
2        1       10326            5           5                1
3        1       12427            1          10               10
4        1       13032            2          10                3


In [ ]:
# Step 9: Prepare temporal product history

temporal_product_history = user_product_history[
    ["user_id", "product_id", "first_order"]
].copy()

print("Temporal product history shape:",
      temporal_product_history.shape)

print("\nMemory usage:",
      round(
          temporal_product_history.memory_usage(deep=True).sum() / 1024**2,
          2
      ),
      "MB")

print("\nSample:")
print(temporal_product_history.head())

Temporal product history shape: (13307953, 3)

Memory usage: 126.91 MB

Sample:
   user_id  product_id  first_order
0        1         196            1
1        1       10258            2
2        1       10326            5
3        1       12427            1
4        1       13032            2


In [ ]:
# Step 10: Load product metadata

products = pd.read_csv(
    f"{base_path}/products.csv",
    dtype={
        "product_id": "int32",
        "aisle_id": "int16",
        "department_id": "int8"
    }
)

print("Products shape:", products.shape)

print("\nColumns:")
print(products.columns.tolist())

print("\nMissing values:")
print(products.isnull().sum())

print("\nUnique products:", products["product_id"].nunique())
print("Unique aisles:", products["aisle_id"].nunique())
print("Unique departments:", products["department_id"].nunique())

print("\nSample:")
print(products.head())

Products shape: (49688, 4)

Columns:
['product_id', 'product_name', 'aisle_id', 'department_id']

Missing values:
product_id       0
product_name     0
aisle_id         0
department_id    0
dtype: int64

Unique products: 49688
Unique aisles: 134
Unique departments: 21

Sample:
   product_id                                       product_name  aisle_id  \
0           1                         Chocolate Sandwich Cookies        61   
1           2                                   All-Seasons Salt       104   
2           3               Robust Golden Unsweetened Oolong Tea        94   
3           4  Smart Ones Classic Favorites Mini Rigatoni Wit...        38   
4           5                          Green Chile Anytime Sauce         5   

   department_id  
0             19  
1             13  
2              7  
3              1  
4             13  


In [ ]:
# Step 11: Add product category information to temporal history

product_category = products[
    ["product_id", "aisle_id", "department_id"]
].copy()

temporal_product_history = temporal_product_history.merge(
    product_category,
    on="product_id",
    how="left"
)

print("Temporal product history shape:",
      temporal_product_history.shape)

print("\nColumns:")
print(temporal_product_history.columns.tolist())

print("\nMissing values:")
print(temporal_product_history.isnull().sum())

print("\nSample:")
print(temporal_product_history.head())

Temporal product history shape: (13307953, 5)

Columns:
['user_id', 'product_id', 'first_order', 'aisle_id', 'department_id']

Missing values:
user_id          0
product_id       0
first_order      0
aisle_id         0
department_id    0
dtype: int64

Sample:
   user_id  product_id  first_order  aisle_id  department_id
0        1         196            1        77              7
1        1       10258            2       117             19
2        1       10326            5        24              4
3        1       12427            1        23             19
4        1       13032            2       121             14


In [ ]:
# Step 12: Calculate global product popularity

product_popularity = (
    prior_interactions
    .groupby("product_id")
    .size()
    .sort_values(ascending=False)
)

TOP_POPULAR_N = 100

popular_products = (
    product_popularity
    .head(TOP_POPULAR_N)
    .index
    .to_numpy(dtype=np.int32)
)

print("Unique products with prior purchases:",
      len(product_popularity))

print("Number of global popular products:",
      len(popular_products))

print("\nTop 20 popular product IDs:")
print(popular_products[:20])

Unique products with prior purchases: 49677
Number of global popular products: 100

Top 20 popular product IDs:
[24852 13176 21137 21903 47209 47766 47626 16797 26209 27845 27966 22935
 24964 45007 39275 49683 28204  5876  8277 40706]


In [ ]:
# Step 13: Calculate department-level product popularity

department_product_popularity = (
    temporal_product_history
    .groupby(["department_id", "product_id"])
    .size()
    .reset_index(name="purchase_count")
)

department_product_popularity = (
    department_product_popularity
    .sort_values(
        ["department_id", "purchase_count"],
        ascending=[True, False]
    )
)

TOP_DEPARTMENT_PRODUCTS = 20

department_popular_products = (
    department_product_popularity
    .groupby("department_id")
    .head(TOP_DEPARTMENT_PRODUCTS)
)

print("Department-product popularity shape:",
      department_product_popularity.shape)

print("Departments:",
      department_product_popularity["department_id"].nunique())

print(
    "Maximum products selected per department:",
    department_popular_products.groupby("department_id").size().max()
)

print("\nSample:")
print(department_popular_products.head(20))

Department-product popularity shape: (49677, 3)
Departments: 21
Maximum products selected per department: 20

Sample:
      department_id  product_id  purchase_count
771               1        9076           23354
1977              1       24489           11715
1704              1       20995           11458
1489              1       17948           11328
3773              1       46802           10650
1237              1       14678            8834
97                1        1158            7760
1418              1       16965            6127
2615              1       32691            6107
3240              1       40545            6081
3955              1       49075            6059
2338              1       28934            6018
2598              1       32433            5928
2969              1       37158            5809
1564              1       18918            5647
3395              1       42450            5257
184               1        2228            4764
1880              

In [ ]:
# Step 14: Build user-level temporal product lookup

user_temporal_products = {}

for user_id, group in temporal_product_history.groupby("user_id"):
    user_temporal_products[user_id] = list(
        zip(
            group["product_id"].astype(int),
            group["first_order"].astype(int)
        )
    )

print("Users in temporal lookup:", len(user_temporal_products))

sample_user = list(user_temporal_products.keys())[0]

print("Sample user:", sample_user)
print(
    "Historical product records:",
    len(user_temporal_products[sample_user])
)

print(
    "First 10 records:",
    user_temporal_products[sample_user][:10]
)

Users in temporal lookup: 206209
Sample user: 1
Historical product records: 18
First 10 records: [(196, 1), (10258, 2), (10326, 5), (12427, 1), (13032, 2), (13176, 2), (14084, 1), (17122, 5), (25133, 3), (26088, 1)]


In [ ]:
# Step 15: Build product popularity by order number

product_order_counts = (
    historical_interactions
    .groupby(
        ["product_id", "order_number"]
    )
    .size()
    .reset_index(name="purchase_count")
)

print("Product-order count shape:",
      product_order_counts.shape)

print("\nUnique products:",
      product_order_counts["product_id"].nunique())

print("Unique order numbers:",
      product_order_counts["order_number"].nunique())

print("\nMemory usage:",
      round(
          product_order_counts.memory_usage(deep=True).sum() / 1024**2,
          2
      ),
      "MB"
      )

print("\nSample:")
print(product_order_counts.head(10))

Product-order count shape: (1778701, 3)

Unique products: 49677
Unique order numbers: 99

Memory usage: 23.75 MB

Sample:
   product_id  order_number  purchase_count
0           1             1              98
1           1             2             103
2           1             3              84
3           1             4              81
4           1             5              84
5           1             6              76
6           1             7              62
7           1             8              62
8           1             9              63
9           1            10              62


In [ ]:
# Step 16: Create department-product popularity by order number

department_product_counts = (
    historical_interactions
    .merge(
        product_category,
        on="product_id",
        how="left"
    )
    .groupby(
        ["department_id", "product_id", "order_number"]
    )
    .size()
    .reset_index(name="purchase_count")
)

print("Department-product-order shape:",
      department_product_counts.shape)

print("\nUnique departments:",
      department_product_counts["department_id"].nunique())

print("Unique products:",
      department_product_counts["product_id"].nunique())

print("Unique order numbers:",
      department_product_counts["order_number"].nunique())

print("\nMemory usage:",
      round(
          department_product_counts.memory_usage(deep=True).sum() / 1024**2,
          2
      ),
      "MB"
)

print("\nSample:")
print(department_product_counts.head(10))

Department-product-order shape: (1778701, 4)

Unique departments: 21
Unique products: 49677
Unique order numbers: 99

Memory usage: 25.44 MB

Sample:
   department_id  product_id  order_number  purchase_count
0              1           4             1              39
1              1           4             2              33
2              1           4             3              44
3              1           4             4              26
4              1           4             5              15
5              1           4             6              17
6              1           4             7              15
7              1           4             8              12
8              1           4             9              21
9              1           4            10              11


In [ ]:
department_product_counts = department_product_counts.sort_values(
    ["department_id", "product_id", "order_number"]
).reset_index(drop=True)

department_product_counts["cumulative_purchase_count"] = (
    department_product_counts
    .groupby(["department_id", "product_id"])["purchase_count"]
    .cumsum()
)

print("Cumulative popularity table created.")
print("Shape:", department_product_counts.shape)
print("\nColumns:")
print(department_product_counts.columns.tolist())
print("\nSample:")
print(department_product_counts.head(10))

Cumulative popularity table created.
Shape: (1778701, 5)

Columns:
['department_id', 'product_id', 'order_number', 'purchase_count', 'cumulative_purchase_count']

Sample:
   department_id  product_id  order_number  purchase_count  \
0              1           4             1              39   
1              1           4             2              33   
2              1           4             3              44   
3              1           4             4              26   
4              1           4             5              15   
5              1           4             6              17   
6              1           4             7              15   
7              1           4             8              12   
8              1           4             9              21   
9              1           4            10              11   

   cumulative_purchase_count  
0                         39  
1                         72  
2                        116  
3                     

In [ ]:
temporal_department_popularity = department_product_counts[
    [
        "department_id",
        "product_id",
        "order_number",
        "cumulative_purchase_count"
    ]
].copy()

print("Temporal department popularity table created.")
print("Shape:", temporal_department_popularity.shape)
print("Memory usage:",
      round(
          temporal_department_popularity.memory_usage(deep=True).sum() / 1024**2,
          2
      ),
      "MB")

print("\nSample:")
print(temporal_department_popularity.head(10))

Temporal department popularity table created.
Shape: (1778701, 4)
Memory usage: 25.44 MB

Sample:
   department_id  product_id  order_number  cumulative_purchase_count
0              1           4             1                         39
1              1           4             2                         72
2              1           4             3                        116
3              1           4             4                        142
4              1           4             5                        157
5              1           4             6                        174
6              1           4             7                        189
7              1           4             8                        201
8              1           4             9                        222
9              1           4            10                        233


In [ ]:
user_department_history = (
    temporal_product_history[
        ["user_id", "first_order", "department_id"]
    ]
    .groupby(["user_id", "department_id"])["first_order"]
    .min()
    .reset_index()
)

user_department_history.columns = [
    "user_id",
    "department_id",
    "first_department_order"
]

print("User-department history created.")
print("Shape:", user_department_history.shape)
print("Unique users:", user_department_history["user_id"].nunique())
print("Unique departments:", user_department_history["department_id"].nunique())

print("\nMemory usage:",
      round(
          user_department_history.memory_usage(deep=True).sum() / 1024**2,
          2
      ),
      "MB")

print("\nSample:")
print(user_department_history.head(10))

User-department history created.
Shape: (2232789, 3)
Unique users: 206209
Unique departments: 21

Memory usage: 14.91 MB

Sample:
   user_id  department_id  first_department_order
0        1              4                       2
1        1              7                       1
2        1             13                       3
3        1             14                       2
4        1             16                       1
5        1             17                       1
6        1             19                       1
7        2              1                       4
8        2              3                      12
9        2              4                       1


In [ ]:
target_order_lookup = temporal_targets[
    [
        "order_id",
        "user_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order"
    ]
].copy()

print("Target-order lookup created.")
print("Shape:", target_order_lookup.shape)
print("Unique target orders:", target_order_lookup["order_id"].nunique())
print("Unique users:", target_order_lookup["user_id"].nunique())

print("\nMemory usage:",
      round(
          target_order_lookup.memory_usage(deep=True).sum() / 1024**2,
          2
      ),
      "MB")

print("\nSample:")
print(target_order_lookup.head(10))

Target-order lookup created.
Shape: (491324, 6)
Unique target orders: 491324
Unique users: 182223

Memory usage: 7.5 MB

Sample:
   order_id  user_id  order_number  order_dow  order_hour_of_day  \
0   2254736        1             4          4                  7   
1    550135        1             7          1                  9   
2   2550362        1            10          4                  8   
3    738281        2             4          2                 10   
4   1718559        2             9          2                  9   
5    839880        2            14          3                 10   
6   2037211        3             4          2                 18   
7   3225766        3             8          0                 17   
8   1402502        3            12          1                 15   
9     94891        4             4          5                 13   

   days_since_prior_order  
0                    29.0  
1                    20.0  
2                    30.0  
3         

In [ ]:
target_products = prior_interactions[
    prior_interactions["order_id"].isin(
        target_order_lookup["order_id"]
    )
].copy()

print("Target products loaded.")
print("Shape:", target_products.shape)
print("Unique target orders:", target_products["order_id"].nunique())
print("Unique products:", target_products["product_id"].nunique())

print("\nMissing values:")
print(target_products.isna().sum())

print("\nSample:")
print(target_products.head(10))

Target products loaded.
Shape: (4982678, 2)
Unique target orders: 491324
Unique products: 46862

Missing values:
order_id      0
product_id    0
dtype: int64

Sample:
    order_id  product_id
56         6       40462
57         6       15873
58         6       41897
62         9       21405
63         9       47890
64         9       11182
65         9        2014
66         9       29193
67         9       34203
68         9       14992


In [ ]:
target_basket_sizes = (
    target_products
    .groupby("order_id")
    .size()
)

print("Target basket-size statistics:")
print(target_basket_sizes.describe())

print("\nMinimum basket size:", target_basket_sizes.min())
print("Median basket size:", target_basket_sizes.median())
print("Mean basket size:", round(target_basket_sizes.mean(), 2))
print("Maximum basket size:", target_basket_sizes.max())

print("\nBasket size distribution:")
print(
    target_basket_sizes
    .value_counts()
    .sort_index()
    .head(20)
)

Target basket-size statistics:
count    491324.000000
mean         10.141328
std           7.627101
min           1.000000
25%           5.000000
50%           8.000000
75%          14.000000
max         145.000000
dtype: float64

Minimum basket size: 1
Median basket size: 8.0
Mean basket size: 10.14
Maximum basket size: 145

Basket size distribution:
1     26308
2     29672
3     31589
4     32975
5     33684
6     33671
7     32448
8     30439
9     27423
10    25319
11    22444
12    20268
13    17810
14    15979
15    14255
16    12397
17    11090
18     9755
19     8422
20     7503
Name: count, dtype: int64


In [ ]:
target_basket_size_lookup = (
    target_products
    .groupby("order_id")
    .size()
    .reset_index(name="target_basket_size")
)

print("Target basket-size lookup created.")
print("Shape:", target_basket_size_lookup.shape)
print("Unique orders:", target_basket_size_lookup["order_id"].nunique())

print("\nMemory usage:",
      round(
          target_basket_size_lookup.memory_usage(deep=True).sum() / 1024**2,
          2
      ),
      "MB")

print("\nSample:")
print(target_basket_size_lookup.head(10))

Target basket-size lookup created.
Shape: (491324, 2)
Unique orders: 491324

Memory usage: 5.62 MB

Sample:
   order_id  target_basket_size
0         6                   3
1         9                  15
2        10                  15
3        11                   5
4        25                  14
5        32                   9
6        39                   5
7        40                   4
8        43                   7
9        51                   9


In [ ]:
target_product_lookup = (
    target_products
    .groupby("order_id")["product_id"]
    .apply(set)
    .to_dict()
)

print("Target-product lookup created.")
print("Number of target orders:", len(target_product_lookup))

sample_order = next(iter(target_product_lookup))

print("\nSample order:", sample_order)
print("Number of products:", len(target_product_lookup[sample_order]))
print("Products:", list(target_product_lookup[sample_order])[:20])

Target-product lookup created.
Number of target orders: 491324

Sample order: 6
Number of products: 3
Products: [15873, 41897, 40462]


In [ ]:
# Step 25: Baseline candidate recall on 5,000 temporal targets

RECALL_SAMPLE_SIZE = 5000

recall_targets = target_order_lookup.head(RECALL_SAMPLE_SIZE)

total_target_products = 0
matched_target_products = 0

for row in recall_targets.itertuples(index=False):
    user_id = row.user_id
    target_order_number = row.order_number
    order_id = row.order_id

    # Products purchased by the user before this target order
    historical_candidates = {
        product_id
        for product_id, first_order in user_temporal_products.get(user_id, [])
        if first_order < target_order_number
    }

    # Add global popular products
    candidate_set = historical_candidates.union(
        popular_products.tolist()
    )

    actual_products = target_product_lookup[order_id]

    total_target_products += len(actual_products)
    matched_target_products += len(
        actual_products.intersection(candidate_set)
    )

baseline_recall = matched_target_products / total_target_products

print("Baseline candidate recall experiment")
print("------------------------------------")
print("Target orders tested:", RECALL_SAMPLE_SIZE)
print("Target products:", total_target_products)
print("Matched products:", matched_target_products)
print("Baseline recall:", round(baseline_recall * 100, 2), "%")

Baseline candidate recall experiment
------------------------------------
Target orders tested: 5000
Target products: 49565
Matched products: 31110
Baseline recall: 62.77 %


In [ ]:
# Step 26: Create cumulative department-product popularity matrix

department_product_matrix = (
    department_product_counts
    .pivot_table(
        index=["department_id", "product_id"],
        columns="order_number",
        values="purchase_count",
        fill_value=0
    )
)

department_product_matrix = department_product_matrix.sort_index(axis=1)

print("Department-product matrix created.")
print("Shape:", department_product_matrix.shape)

print("\nOrder-number range:",
      department_product_matrix.columns.min(),
      "to",
      department_product_matrix.columns.max())

print("\nMemory usage:",
      round(
          department_product_matrix.memory_usage(deep=True).sum() / 1024**2,
          2
      ),
      "MB")

print("\nSample:")
print(department_product_matrix.iloc[:5, :10])

Department-product matrix created.
Shape: (49677, 99)

Order-number range: 1 to 99

Memory usage: 37.95 MB

Sample:
order_number                1     2     3     4     5     6     7     8   \
department_id product_id                                                   
1             4           39.0  33.0  44.0  26.0  15.0  17.0  15.0  12.0   
              8           14.0  15.0  11.0  10.0   8.0   8.0   7.0   5.0   
              12          21.0  13.0  18.0  18.0  12.0  12.0  16.0   7.0   
              18           8.0  16.0   7.0  11.0   8.0   7.0   4.0   6.0   
              30           8.0   6.0   6.0   7.0   7.0   3.0   6.0   1.0   

order_number                9     10  
department_id product_id              
1             4           21.0  11.0  
              8            6.0   6.0  
              12          12.0   7.0  
              18           3.0   4.0  
              30           2.0   3.0  


In [ ]:
# Step 27: Convert order-level counts to cumulative counts

department_product_cumulative = department_product_matrix.cumsum(axis=1)

print("Cumulative department-product matrix created.")
print("Shape:", department_product_cumulative.shape)

print("\nMemory usage:",
      round(
          department_product_cumulative.memory_usage(deep=True).sum() / 1024**2,
          2
      ),
      "MB")

print("\nSample:")
print(department_product_cumulative.iloc[:5, :10])

Cumulative department-product matrix created.
Shape: (49677, 99)

Memory usage: 37.95 MB

Sample:
order_number                1     2      3      4      5      6      7   \
department_id product_id                                                  
1             4           39.0  72.0  116.0  142.0  157.0  174.0  189.0   
              8           14.0  29.0   40.0   50.0   58.0   66.0   73.0   
              12          21.0  34.0   52.0   70.0   82.0   94.0  110.0   
              18           8.0  24.0   31.0   42.0   50.0   57.0   61.0   
              30           8.0  14.0   20.0   27.0   34.0   37.0   43.0   

order_number                 8      9      10  
department_id product_id                       
1             4           201.0  222.0  233.0  
              8            78.0   84.0   90.0  
              12          117.0  129.0  136.0  
              18           67.0   70.0   74.0  
              30           44.0   46.0   49.0  


In [ ]:
# Step 28: Create temporally valid top-20 products per department

TOP_DEPARTMENT_CANDIDATES = 20

temporal_department_candidates = {}

for department_id in department_product_cumulative.index.get_level_values(
    "department_id"
).unique():

    department_data = department_product_cumulative.loc[department_id]

    for order_number in range(2, 100):
        # Only information available BEFORE this order
        historical_order = order_number - 1

        popularity = department_data[historical_order]

        top_products = (
            popularity[popularity > 0]
            .sort_values(ascending=False)
            .head(TOP_DEPARTMENT_CANDIDATES)
            .index
            .astype(np.int32)
            .tolist()
        )

        temporal_department_candidates[
            (int(department_id), order_number)
        ] = top_products

print("Temporal department candidate lookup created.")
print("Number of lookup entries:", len(temporal_department_candidates))

sample_key = (1, 4)

print("\nSample lookup:", sample_key)
print(
    "Top products before order 4 in department 1:"
)
print(temporal_department_candidates[sample_key])

print("\nCandidates in sample:", len(temporal_department_candidates[sample_key]))

Temporal department candidate lookup created.
Number of lookup entries: 2058

Sample lookup: (1, 4)
Top products before order 4 in department 1:
[9076, 24489, 17948, 20995, 46802, 1158, 37158, 42450, 14678, 28934, 40545, 7963, 16965, 49075, 24561, 32691, 20345, 2228, 18918, 32433]

Candidates in sample: 20


In [ ]:
# Step 29: Prepare user-department interaction counts

user_department_counts = (
    historical_interactions
    .merge(
        product_category,
        on="product_id",
        how="left"
    )
    .groupby(
        ["user_id", "department_id", "order_number"]
    )
    .size()
    .reset_index(name="purchase_count")
)

print("User-department-order counts created.")
print("Shape:", user_department_counts.shape)

print("Unique users:", user_department_counts["user_id"].nunique())
print("Unique departments:", user_department_counts["department_id"].nunique())
print("Unique order numbers:", user_department_counts["order_number"].nunique())

print("\nMemory usage:",
      round(
          user_department_counts.memory_usage(deep=True).sum() / 1024**2,
          2
      ),
      "MB")

print("\nSample:")
print(user_department_counts.head(10))

User-department-order counts created.
Shape: (15226198, 4)
Unique users: 206209
Unique departments: 21
Unique order numbers: 99

Memory usage: 217.81 MB

Sample:
   user_id  department_id  order_number  purchase_count
0        1              4             2               1
1        1              4             5               4
2        1              7             1               1
3        1              7             2               1
4        1              7             3               1
5        1              7             4               1
6        1              7             5               1
7        1              7             6               1
8        1              7             7               1
9        1              7             8               2


In [ ]:
# Step 30: Create cumulative user-department purchase counts

user_department_counts = user_department_counts.sort_values(
    ["user_id", "department_id", "order_number"]
).reset_index(drop=True)

user_department_counts["cumulative_purchase_count"] = (
    user_department_counts
    .groupby(["user_id", "department_id"])["purchase_count"]
    .cumsum()
)

print("Cumulative user-department counts created.")
print("Shape:", user_department_counts.shape)

print("\nColumns:")
print(user_department_counts.columns.tolist())

print("\nSample:")
print(user_department_counts.head(10))

Cumulative user-department counts created.
Shape: (15226198, 5)

Columns:
['user_id', 'department_id', 'order_number', 'purchase_count', 'cumulative_purchase_count']

Sample:
   user_id  department_id  order_number  purchase_count  \
0        1              4             2               1   
1        1              4             5               4   
2        1              7             1               1   
3        1              7             2               1   
4        1              7             3               1   
5        1              7             4               1   
6        1              7             5               1   
7        1              7             6               1   
8        1              7             7               1   
9        1              7             8               2   

   cumulative_purchase_count  
0                          1  
1                          5  
2                          1  
3                          2  
4                   

In [ ]:
# Step 31: Find each user's top 3 departments before every target order

target_user_dept = target_order_lookup[
    ["order_id", "user_id", "order_number"]
].merge(
    user_department_counts,
    on="user_id",
    how="left"
)

# Keep only department information available before the target order
target_user_dept = target_user_dept[
    target_user_dept["order_number_y"] < target_user_dept["order_number_x"]
].copy()

# For each target, keep the latest cumulative count available for each department
target_user_dept = (
    target_user_dept
    .sort_values(
        ["order_id", "department_id", "order_number_y"]
    )
    .groupby(
        ["order_id", "department_id"],
        as_index=False
    )
    .tail(1)
)

# Select the top 3 departments for each target
target_user_dept = (
    target_user_dept
    .sort_values(
        ["order_id", "cumulative_purchase_count"],
        ascending=[True, False]
    )
    .groupby("order_id")
    .head(3)
)

print("Top-3 user departments created.")
print("Shape:", target_user_dept.shape)
print("Unique target orders:", target_user_dept["order_id"].nunique())

print("\nSample:")
print(
    target_user_dept[
        [
            "order_id",
            "user_id",
            "department_id",
            "cumulative_purchase_count"
        ]
    ].head(15)
)

Top-3 user departments created.
Shape: (1456252, 7)
Unique target orders: 491324

Sample:
          order_id  user_id  department_id  cumulative_purchase_count
4705251          6    22352              4                         10
4705258          6    22352              7                          7
4705271          6    22352             16                          3
29418155         9   139016              4                         48
29418246         9   139016             16                         25
29418139         9   139016              3                         22
28662545        10   135442              4                         14
28662580        10   135442             12                          5
28662633        10   135442             18                          4
30423940        11   143742             13                          5
30423900        11   143742              1                          4
30423933        11   143742              9                          2


In [ ]:
# Step 32: Keep only the required top-department information

target_user_dept = target_user_dept[
    [
        "order_id",
        "user_id",
        "department_id",
        "cumulative_purchase_count"
    ]
].copy()

target_user_dept = target_user_dept.sort_values(
    ["order_id", "cumulative_purchase_count"],
    ascending=[True, False]
).reset_index(drop=True)

print("Clean top-department table created.")
print("Shape:", target_user_dept.shape)
print("Unique target orders:", target_user_dept["order_id"].nunique())
print("Unique users:", target_user_dept["user_id"].nunique())

print("\nMemory usage:",
      round(
          target_user_dept.memory_usage(deep=True).sum() / 1024**2,
          2
      ),
      "MB")

print("\nDepartments per target:")
print(
    target_user_dept
    .groupby("order_id")
    .size()
    .value_counts()
    .sort_index()
)

print("\nSample:")
print(target_user_dept.head(15))

Clean top-department table created.
Shape: (1456252, 4)
Unique target orders: 491324
Unique users: 182223

Memory usage: 23.61 MB

Departments per target:
1      4298
2      9124
3    477902
Name: count, dtype: int64

Sample:
    order_id  user_id  department_id  cumulative_purchase_count
0          6    22352              4                         10
1          6    22352              7                          7
2          6    22352             16                          3
3          9   139016              4                         48
4          9   139016             16                         25
5          9   139016              3                         22
6         10   135442              4                         14
7         10   135442             12                          5
8         10   135442             18                          4
9         11   143742             13                          5
10        11   143742              1                          4
11    

In [ ]:
# Step 33: Create a fast lookup of top departments for each target order

target_department_lookup = (
    target_user_dept
    .groupby("order_id")["department_id"]
    .apply(lambda x: x.astype(int).tolist())
    .to_dict()
)

print("Target-department lookup created.")
print("Number of target orders:", len(target_department_lookup))

sample_order = 6

print("\nSample target order:", sample_order)
print(
    "Top departments:",
    target_department_lookup[sample_order]
)

print(
    "\nNumber of departments for sample:",
    len(target_department_lookup[sample_order])
)

Target-department lookup created.
Number of target orders: 491324

Sample target order: 6
Top departments: [4, 7, 16]

Number of departments for sample: 3


In [ ]:
# Step 34: Department-aware candidate recall on 5,000 targets

department_recall_sample = target_order_lookup.head(5000)

total_target_products = 0
matched_target_products = 0
total_candidates = 0

for row in department_recall_sample.itertuples(index=False):

    order_id = row.order_id
    target_order_number = row.order_number
    user_id = row.user_id

    # 1. User's historical products before the target order
    historical_candidates = {
        product_id
        for product_id, first_order in user_temporal_products.get(user_id, [])
        if first_order < target_order_number
    }

    # 2. Global popular products
    candidate_set = historical_candidates.union(
        popular_products.tolist()
    )

    # 3. User's top departments before the target
    user_departments = target_department_lookup.get(order_id, [])

    # 4. Add temporally valid top products from those departments
    for department_id in user_departments:
        candidate_set.update(
            temporal_department_candidates.get(
                (department_id, target_order_number),
                []
            )
        )

    actual_products = target_product_lookup[order_id]

    total_target_products += len(actual_products)
    matched_target_products += len(
        actual_products.intersection(candidate_set)
    )

    total_candidates += len(candidate_set)

department_recall = (
    matched_target_products / total_target_products
)

print("Department-aware candidate recall experiment")
print("---------------------------------------------")
print("Target orders tested:", len(department_recall_sample))
print("Target products:", total_target_products)
print("Matched products:", matched_target_products)
print("Recall:", round(department_recall * 100, 2), "%")
print("Average candidates per target:",
      round(total_candidates / len(department_recall_sample), 2))
print("Recall improvement:",
      round((department_recall - baseline_recall) * 100, 2),
      "percentage points")

Department-aware candidate recall experiment
---------------------------------------------
Target orders tested: 5000
Target products: 49565
Matched products: 31513
Recall: 63.58 %
Average candidates per target: 167.3
Recall improvement: 0.81 percentage points


In [ ]:
# Step 35: Build temporally valid user-aisle history

user_aisle_history = (
    temporal_product_history[
        ["user_id", "product_id", "first_order", "aisle_id"]
    ]
    .groupby(["user_id", "aisle_id"])
    .agg(
        first_aisle_order=("first_order", "min"),
        unique_products=("product_id", "nunique")
    )
    .reset_index()
)

print("User-aisle history created.")
print("Shape:", user_aisle_history.shape)
print("Unique users:", user_aisle_history["user_id"].nunique())
print("Unique aisles:", user_aisle_history["aisle_id"].nunique())

print("\nMemory usage:",
      round(
          user_aisle_history.memory_usage(deep=True).sum() / 1024**2,
          2
      ),
      "MB")

print("\nSample:")
print(user_aisle_history.head(10))


User-aisle history created.
Shape: (5729249, 4)
Unique users: 206209
Unique aisles: 134

Memory usage: 87.42 MB

Sample:
   user_id  aisle_id  first_aisle_order  unique_products
0        1        21                  3                1
1        1        23                  1                2
2        1        24                  2                4
3        1        45                 10                1
4        1        53                  8                1
5        1        54                  1                1
6        1        77                  1                2
7        1        88                  3                1
8        1        91                  1                2
9        1       117                  2                1


In [ ]:
# Step 36: Find top 3 aisles for each target user

target_user_aisles = target_order_lookup[
    ["order_id", "user_id", "order_number"]
].merge(
    user_aisle_history,
    on="user_id",
    how="left"
)

# Keep only aisles known before the target order
target_user_aisles = target_user_aisles[
    target_user_aisles["first_aisle_order"] < target_user_aisles["order_number"]
].copy()

# Select the top 3 aisles based on number of unique products
target_user_aisles = (
    target_user_aisles
    .sort_values(
        ["order_id", "unique_products", "first_aisle_order"],
        ascending=[True, False, True]
    )
    .groupby("order_id")
    .head(3)
)

print("Top-3 user aisles created.")
print("Shape:", target_user_aisles.shape)
print("Unique target orders:", target_user_aisles["order_id"].nunique())
print("Unique users:", target_user_aisles["user_id"].nunique())

print("\nAisles per target:")
print(
    target_user_aisles
    .groupby("order_id")
    .size()
    .value_counts()
    .sort_index()
)

print("\nSample:")
print(
    target_user_aisles[
        [
            "order_id",
            "user_id",
            "aisle_id",
            "unique_products",
            "first_aisle_order"
        ]
    ].head(15)
)

Top-3 user aisles created.
Shape: (1465781, 6)
Unique target orders: 491324
Unique users: 182223

Aisles per target:
1      2044
2      4103
3    485177
Name: count, dtype: int64

Sample:
          order_id  user_id  aisle_id  unique_products  first_aisle_order
1638976          6    22352        83                8                  2
1638963          6    22352        24                6                  2
1638962          6    22352        21                5                  2
10253644         9   139016        83               19                  1
10253666         9   139016       123                6                  1
10253656         9   139016       107                6                  2
9990479         10   135442        83               18                  1
9990470         10   135442        24                9                  1
9990482         10   135442        92                7                  3
10601193        11   143742       105                3                  

In [ ]:
# Step 37: Create aisle-product-order purchase counts

aisle_product_counts = (
    product_order_counts
    .merge(
        products[["product_id", "aisle_id"]],
        on="product_id",
        how="left"
    )
)

aisle_product_counts = aisle_product_counts[
    [
        "aisle_id",
        "product_id",
        "order_number",
        "purchase_count"
    ]
]

print("Aisle-product-order counts created.")
print("Shape:", aisle_product_counts.shape)

print("Unique aisles:", aisle_product_counts["aisle_id"].nunique())
print("Unique products:", aisle_product_counts["product_id"].nunique())
print("Unique order numbers:", aisle_product_counts["order_number"].nunique())

print("\nMemory usage:",
      round(
          aisle_product_counts.memory_usage(deep=True).sum() / 1024**2,
          2
      ),
      "MB")

print("\nMissing values:")
print(aisle_product_counts.isna().sum())

print("\nSample:")
print(aisle_product_counts.head(10))

Aisle-product-order counts created.
Shape: (1778701, 4)
Unique aisles: 134
Unique products: 49677
Unique order numbers: 99

Memory usage: 27.14 MB

Missing values:
aisle_id          0
product_id        0
order_number      0
purchase_count    0
dtype: int64

Sample:
   aisle_id  product_id  order_number  purchase_count
0        61           1             1              98
1        61           1             2             103
2        61           1             3              84
3        61           1             4              81
4        61           1             5              84
5        61           1             6              76
6        61           1             7              62
7        61           1             8              62
8        61           1             9              63
9        61           1            10              62


In [ ]:
# Step 38: Create cumulative aisle-product popularity

aisle_product_counts = aisle_product_counts.sort_values(
    ["aisle_id", "product_id", "order_number"]
).reset_index(drop=True)

aisle_product_counts["cumulative_purchase_count"] = (
    aisle_product_counts
    .groupby(["aisle_id", "product_id"])["purchase_count"]
    .cumsum()
)

print("Cumulative aisle-product popularity created.")
print("Shape:", aisle_product_counts.shape)

print("\nColumns:")
print(aisle_product_counts.columns.tolist())

print("\nMemory usage:",
      round(
          aisle_product_counts.memory_usage(deep=True).sum() / 1024**2,
          2
      ),
      "MB")

print("\nSample:")
print(aisle_product_counts.head(10))

Cumulative aisle-product popularity created.
Shape: (1778701, 5)

Columns:
['aisle_id', 'product_id', 'order_number', 'purchase_count', 'cumulative_purchase_count']

Memory usage: 40.71 MB

Sample:
   aisle_id  product_id  order_number  purchase_count  \
0         1         209             1               9   
1         1         209             2               9   
2         1         209             3               9   
3         1         209             4               5   
4         1         209             5               9   
5         1         209             6               4   
6         1         209             7               8   
7         1         209             8               4   
8         1         209             9               3   
9         1         209            10               1   

   cumulative_purchase_count  
0                          9  
1                         18  
2                         27  
3                         32  
4                  

In [ ]:
# Step 39: Create temporally valid top-20 products per aisle

TOP_AISLE_CANDIDATES = 20

temporal_aisle_candidates = {}

for aisle_id in aisle_product_counts["aisle_id"].unique():

    aisle_data = aisle_product_counts[
        aisle_product_counts["aisle_id"] == aisle_id
    ]

    for order_number in range(2, 100):

        historical_data = aisle_data[
            aisle_data["order_number"] == order_number - 1
        ]

        top_products = (
            historical_data
            .sort_values(
                "cumulative_purchase_count",
                ascending=False
            )
            .head(TOP_AISLE_CANDIDATES)["product_id"]
            .astype(np.int32)
            .tolist()
        )

        temporal_aisle_candidates[
            (int(aisle_id), order_number)
        ] = top_products

print("Temporal aisle candidate lookup created.")
print("Number of lookup entries:", len(temporal_aisle_candidates))

sample_key = (1, 4)

print("\nSample lookup:", sample_key)
print(
    "Top products before order 4 in aisle 1:"
)
print(temporal_aisle_candidates[sample_key])

print("\nCandidates in sample:",
      len(temporal_aisle_candidates[sample_key]))

Temporal aisle candidate lookup created.
Number of lookup entries: 13132

Sample lookup: (1, 4)
Top products before order 4 in aisle 1:
[25199, 26047, 22281, 21560, 23719, 29180, 43221, 25965, 26714, 47329, 15767, 28698, 554, 47885, 36878, 18923, 8382, 11888, 26352, 45757]

Candidates in sample: 20


In [ ]:
# Step 40: Evaluate aisle-aware candidate recall

SAMPLE_TARGETS = temporal_targets.head(5000)

total_target_products = 0
matched_target_products = 0
total_candidates = 0

for _, target in SAMPLE_TARGETS.iterrows():

    order_id = int(target["order_id"])
    user_id = int(target["user_id"])
    order_number = int(target["order_number"])

    # Target basket
    actual_products = target_product_lookup[order_id]

    # 1. User historical products before target
    user_history = {
        product_id
        for product_id, first_order in user_temporal_products[user_id]
        if first_order < order_number
    }

    # 2. Global popular products
    candidates = set(user_history)
    candidates.update(popular_products)

    # 3. Top products from user's top departments
    for department_id in target_department_lookup.get(order_id, []):
        candidates.update(
            temporal_department_candidates.get(
                (department_id, order_number), []
            )
        )

    # 4. Top products from user's top aisles
    user_aisles = (
        target_user_aisles[
            target_user_aisles["order_id"] == order_id
        ]["aisle_id"]
        .astype(int)
        .tolist()
    )

    for aisle_id in user_aisles:
        candidates.update(
            temporal_aisle_candidates.get(
                (aisle_id, order_number), []
            )
        )

    # Recall calculation
    matched = len(set(actual_products) & candidates)

    total_target_products += len(actual_products)
    matched_target_products += matched
    total_candidates += len(candidates)

recall = (
    matched_target_products / total_target_products
    if total_target_products > 0 else 0
)

average_candidates = total_candidates / len(SAMPLE_TARGETS)

print("Aisle-aware candidate evaluation completed.")
print("Target orders:", len(SAMPLE_TARGETS))
print("Target products:", total_target_products)
print("Matched target products:", matched_target_products)
print("Recall:", round(recall * 100, 2), "%")
print("Average candidates per target:", round(average_candidates, 2))

Aisle-aware candidate evaluation completed.
Target orders: 5000
Target products: 49565
Matched target products: 32152
Recall: 64.87 %
Average candidates per target: 195.67


In [ ]:
# Step 41: Build combined temporal candidate lookup efficiently

# Create order -> top aisles lookup once
target_aisle_lookup = (
    target_user_aisles
    .groupby("order_id")["aisle_id"]
    .apply(lambda x: x.astype(int).tolist())
    .to_dict()
)

combined_candidate_lookup = {}

for row in target_order_lookup[
    ["order_id", "user_id", "order_number"]
].itertuples(index=False):

    order_id = int(row.order_id)
    user_id = int(row.user_id)
    order_number = int(row.order_number)

    candidates = set()

    # 1. User historical products before target
    for product_id, first_order in user_temporal_products[user_id]:
        if first_order < order_number:
            candidates.add(product_id)

    # 2. Global popular products
    candidates.update(popular_products.tolist())

    # 3. User's top departments
    for department_id in target_department_lookup.get(order_id, []):
        candidates.update(
            temporal_department_candidates.get(
                (department_id, order_number), []
            )
        )

    # 4. User's top aisles
    for aisle_id in target_aisle_lookup.get(order_id, []):
        candidates.update(
            temporal_aisle_candidates.get(
                (aisle_id, order_number), []
            )
        )

    combined_candidate_lookup[order_id] = np.array(
        list(candidates),
        dtype=np.int32
    )

print("Combined candidate lookup created.")
print("Target orders:", len(combined_candidate_lookup))

candidate_counts = np.array([
    len(x) for x in combined_candidate_lookup.values()
])

print("\nCandidate statistics:")
print("Mean:", round(candidate_counts.mean(), 2))
print("Median:", round(np.median(candidate_counts), 2))
print("Minimum:", candidate_counts.min())
print("Maximum:", candidate_counts.max())

sample_order = next(iter(combined_candidate_lookup))

print("\nSample order:", sample_order)
print("Number of candidates:",
      len(combined_candidate_lookup[sample_order]))

print("First 20 candidates:",
      combined_candidate_lookup[sample_order][:20])

Combined candidate lookup created.
Target orders: 491324

Candidate statistics:
Mean: 196.02
Median: 188.0
Minimum: 100
Maximum: 797

Sample order: 2254736
Number of candidates: 179
First 20 candidates: [20995 42500 47626 45066  3599 10258 49683 22035 27156 46616  9755 15902
 18465 47141 48679 28199 47144  4138 28204 25133]


In [ ]:
# Step 42: Verify temporal leakage in combined candidates

violations = 0
checked_candidates = 0

for order_id, candidates in list(combined_candidate_lookup.items())[:5000]:

    target_info = target_order_lookup[
        target_order_lookup["order_id"] == order_id
    ].iloc[0]

    user_id = int(target_info["user_id"])
    target_order_number = int(target_info["order_number"])

    # Historical products must have appeared before the target
    historical_products = {
        product_id
        for product_id, first_order in user_temporal_products[user_id]
        if first_order < target_order_number
    }

    for product_id in candidates:

        # Candidate is allowed if it comes from history,
        # or from a popularity source.
        if product_id in historical_products:
            checked_candidates += 1

            # Verify first purchase happened before target
            first_order = next(
                first
                for prod, first in user_temporal_products[user_id]
                if prod == product_id
            )

            if first_order >= target_order_number:
                violations += 1

print("Leakage verification completed.")
print("Target orders checked:", 5000)
print("Historical candidates checked:", checked_candidates)
print("Temporal violations:", violations)

if violations == 0:
    print("STATUS: No temporal leakage detected.")
else:
    print("STATUS: Temporal leakage detected.")

Leakage verification completed.
Target orders checked: 5000
Historical candidates checked: 231796
Temporal violations: 0
STATUS: No temporal leakage detected.


In [ ]:
# Step 43: Prepare chunked training-data generation

candidate_output_path = f"{feature_path}/Candidate_Training_Data"

os.makedirs(candidate_output_path, exist_ok=True)

CHUNK_SIZE = 5000

total_targets = len(target_order_lookup)
total_chunks = int(np.ceil(total_targets / CHUNK_SIZE))

print("Candidate training-data directory:", candidate_output_path)
print("Total target orders:", total_targets)
print("Chunk size:", CHUNK_SIZE)
print("Total chunks:", total_chunks)

print("\nEstimated candidate rows:")
print(
    "Approximately",
    round(candidate_counts.mean() * total_targets / 1_000_000, 2),
    "million rows"
)

Candidate training-data directory: /content/drive/MyDrive/ML_Market_Basket_Analysis/02_Datasets/Features/Candidate_Training_Data
Total target orders: 491324
Chunk size: 5000
Total chunks: 99

Estimated candidate rows:
Approximately 96.31 million rows


In [ ]:
# Step 44: Generate the first candidate-training chunk

chunk_number = 1

chunk_targets = target_order_lookup.iloc[
    0:CHUNK_SIZE
].copy()

chunk_rows = []

for row in chunk_targets.itertuples(index=False):

    order_id = int(row.order_id)
    user_id = int(row.user_id)
    order_number = int(row.order_number)

    actual_products = target_product_lookup[order_id]

    candidates = combined_candidate_lookup[order_id]

    for product_id in candidates:

        chunk_rows.append({
            "order_id": order_id,
            "user_id": user_id,
            "order_number": order_number,
            "product_id": int(product_id),
            "in_next_basket": int(product_id in actual_products)
        })

chunk_1 = pd.DataFrame(chunk_rows)

print("Chunk 1 generated.")
print("Shape:", chunk_1.shape)

print("\nColumns:")
print(chunk_1.columns.tolist())

print("\nPositive labels:",
      int(chunk_1["in_next_basket"].sum()))

print("Negative labels:",
      int((chunk_1["in_next_basket"] == 0).sum()))

print("\nPositive rate:",
      round(chunk_1["in_next_basket"].mean() * 100, 2), "%")

print("\nUnique target orders:",
      chunk_1["order_id"].nunique())

print("\nAverage candidates per order:",
      round(len(chunk_1) / chunk_1["order_id"].nunique(), 2))

print("\nSample:")
print(chunk_1.head(10))

Chunk 1 generated.
Shape: (978360, 5)

Columns:
['order_id', 'user_id', 'order_number', 'product_id', 'in_next_basket']

Positive labels: 32152
Negative labels: 946208

Positive rate: 3.29 %

Unique target orders: 5000

Average candidates per order: 195.67

Sample:
   order_id  user_id  order_number  product_id  in_next_basket
0   2254736        1             4       20995               0
1   2254736        1             4       42500               0
2   2254736        1             4       47626               0
3   2254736        1             4       45066               0
4   2254736        1             4        3599               0
5   2254736        1             4       10258               1
6   2254736        1             4       49683               0
7   2254736        1             4       22035               0
8   2254736        1             4       27156               0
9   2254736        1             4       46616               0


In [ ]:
# Step 45: Validate candidate labels

positive_rows = chunk_1[chunk_1["in_next_basket"] == 1]

label_errors = 0

for row in positive_rows.itertuples(index=False):

    actual_products = target_product_lookup[row.order_id]

    if row.product_id not in actual_products:
        label_errors += 1

print("Positive-label validation completed.")
print("Positive rows checked:", len(positive_rows))
print("Label errors:", label_errors)

if label_errors == 0:
    print("STATUS: All positive labels are correct.")
else:
    print("STATUS: Label errors detected.")

# Check how many target-basket products were NOT candidates
missing_target_products = 0

for order_id in chunk_1["order_id"].unique():

    actual_products = target_product_lookup[order_id]

    candidate_products = set(
        chunk_1.loc[
            chunk_1["order_id"] == order_id,
            "product_id"
        ]
    )

    missing_target_products += len(
        set(actual_products) - candidate_products
    )

print("\nTarget products missing from candidate sets:",
      missing_target_products)

Positive-label validation completed.
Positive rows checked: 32152
Label errors: 0
STATUS: All positive labels are correct.

Target products missing from candidate sets: 17413


In [ ]:
# Step 46: Analyze missing target products

missing_analysis = []

for order_id in chunk_1["order_id"].unique():

    target_info = target_order_lookup[
        target_order_lookup["order_id"] == order_id
    ].iloc[0]

    user_id = int(target_info["user_id"])
    order_number = int(target_info["order_number"])

    actual_products = target_product_lookup[order_id]
    candidate_products = set(
        chunk_1.loc[
            chunk_1["order_id"] == order_id,
            "product_id"
        ]
    )

    missing_products = set(actual_products) - candidate_products

    history_products = {
        product_id
        for product_id, first_order in user_temporal_products[user_id]
        if first_order < order_number
    }

    for product_id in missing_products:

        missing_analysis.append({
            "order_id": order_id,
            "user_id": user_id,
            "product_id": product_id,
            "was_previously_purchased": product_id in history_products
        })

missing_df = pd.DataFrame(missing_analysis)

print("Missing-product analysis completed.")
print("Missing target products:", len(missing_df))

print("\nPreviously purchased:")
print(
    missing_df["was_previously_purchased"]
    .value_counts()
)

print("\nPercentage:")
print(
    (missing_df["was_previously_purchased"]
     .value_counts(normalize=True) * 100)
    .round(2)
)

Missing-product analysis completed.
Missing target products: 17413

Previously purchased:
was_previously_purchased
False    17413
Name: count, dtype: int64

Percentage:
was_previously_purchased
False    100.0
Name: proportion, dtype: float64


In [ ]:
# Step 47: Test larger global popularity pools

SAMPLE_TARGETS = target_order_lookup.head(5000)

global_top_sizes = [100, 500, 1000]

for top_n in global_top_sizes:

    global_candidates = (
        product_popularity
        .head(top_n)
        .index
        .to_numpy(dtype=np.int32)
    )

    total_target_products = 0
    matched_target_products = 0
    total_candidates = 0

    for row in SAMPLE_TARGETS.itertuples(index=False):

        order_id = int(row.order_id)
        user_id = int(row.user_id)
        order_number = int(row.order_number)

        actual_products = target_product_lookup[order_id]

        candidates = set()

        # User history
        for product_id, first_order in user_temporal_products[user_id]:
            if first_order < order_number:
                candidates.add(product_id)

        # Global popularity
        candidates.update(global_candidates.tolist())

        # Department candidates
        for department_id in target_department_lookup.get(order_id, []):
            candidates.update(
                temporal_department_candidates.get(
                    (department_id, order_number), []
                )
            )

        # Aisle candidates
        for aisle_id in target_aisle_lookup.get(order_id, []):
            candidates.update(
                temporal_aisle_candidates.get(
                    (aisle_id, order_number), []
                )
            )

        matched = len(set(actual_products) & candidates)

        total_target_products += len(actual_products)
        matched_target_products += matched
        total_candidates += len(candidates)

    recall = matched_target_products / total_target_products
    avg_candidates = total_candidates / len(SAMPLE_TARGETS)

    print(
        f"Global Top-{top_n}: "
        f"Recall = {recall * 100:.2f}% | "
        f"Avg candidates = {avg_candidates:.2f}"
    )

Global Top-100: Recall = 64.87% | Avg candidates = 195.67
Global Top-500: Recall = 71.22% | Avg candidates = 553.75
Global Top-1000: Recall = 75.69% | Avg candidates = 1039.27


In [ ]:
# Step 48: Find the global popularity sweet spot

SAMPLE_TARGETS = target_order_lookup.head(5000)

global_top_sizes = [200, 300]

for top_n in global_top_sizes:

    global_candidates = (
        product_popularity
        .head(top_n)
        .index
        .to_numpy(dtype=np.int32)
    )

    total_target_products = 0
    matched_target_products = 0
    total_candidates = 0

    for row in SAMPLE_TARGETS.itertuples(index=False):

        order_id = int(row.order_id)
        user_id = int(row.user_id)
        order_number = int(row.order_number)

        actual_products = target_product_lookup[order_id]

        candidates = set()

        # User history
        for product_id, first_order in user_temporal_products[user_id]:
            if first_order < order_number:
                candidates.add(product_id)

        # Global popularity
        candidates.update(global_candidates.tolist())

        # Department candidates
        for department_id in target_department_lookup.get(order_id, []):
            candidates.update(
                temporal_department_candidates.get(
                    (department_id, order_number), []
                )
            )

        # Aisle candidates
        for aisle_id in target_aisle_lookup.get(order_id, []):
            candidates.update(
                temporal_aisle_candidates.get(
                    (aisle_id, order_number), []
                )
            )

        matched = len(set(actual_products) & candidates)

        total_target_products += len(actual_products)
        matched_target_products += matched
        total_candidates += len(candidates)

    recall = matched_target_products / total_target_products
    avg_candidates = total_candidates / len(SAMPLE_TARGETS)

    print(
        f"Global Top-{top_n}: "
        f"Recall = {recall * 100:.2f}% | "
        f"Avg candidates = {avg_candidates:.2f}"
    )

Global Top-200: Recall = 67.15% | Avg candidates = 281.80
Global Top-300: Recall = 68.69% | Avg candidates = 369.99


In [ ]:
# Step 49: Set final global candidate pool

TOP_GLOBAL_CANDIDATES = 500

global_candidate_products = (
    product_popularity
    .head(TOP_GLOBAL_CANDIDATES)
    .index
    .to_numpy(dtype=np.int32)
)

print("Final global candidate pool created.")
print("Number of global candidates:",
      len(global_candidate_products))

print("\nTop 20 global candidates:")
print(global_candidate_products[:20])

print("\nLowest-ranked global candidate:",
      global_candidate_products[-1])

Final global candidate pool created.
Number of global candidates: 500

Top 20 global candidates:
[24852 13176 21137 21903 47209 47766 47626 16797 26209 27845 27966 22935
 24964 45007 39275 49683 28204  5876  8277 40706]

Lowest-ranked global candidate: 11712


In [ ]:
# Step 50: Rebuild combined temporal candidates using Top-500

combined_candidate_lookup = {}

for row in target_order_lookup[
    ["order_id", "user_id", "order_number"]
].itertuples(index=False):

    order_id = int(row.order_id)
    user_id = int(row.user_id)
    order_number = int(row.order_number)

    candidates = set()

    # 1. User historical products before target
    for product_id, first_order in user_temporal_products[user_id]:
        if first_order < order_number:
            candidates.add(product_id)

    # 2. Global Top-500 products
    candidates.update(global_candidate_products.tolist())

    # 3. User's top departments
    for department_id in target_department_lookup.get(order_id, []):
        candidates.update(
            temporal_department_candidates.get(
                (department_id, order_number), []
            )
        )

    # 4. User's top aisles
    for aisle_id in target_aisle_lookup.get(order_id, []):
        candidates.update(
            temporal_aisle_candidates.get(
                (aisle_id, order_number), []
            )
        )

    combined_candidate_lookup[order_id] = np.array(
        list(candidates),
        dtype=np.int32
    )

candidate_counts = np.array([
    len(x) for x in combined_candidate_lookup.values()
])

print("Top-500 combined candidate lookup created.")
print("Target orders:", len(combined_candidate_lookup))

print("\nCandidate statistics:")
print("Mean:", round(candidate_counts.mean(), 2))
print("Median:", round(np.median(candidate_counts), 2))
print("Minimum:", candidate_counts.min())
print("Maximum:", candidate_counts.max())

print("\nEstimated total candidate rows:",
      f"{candidate_counts.sum():,}")

print("Estimated size:",
      round(candidate_counts.sum() / 1_000_000, 2),
      "million rows")

Top-500 combined candidate lookup created.
Target orders: 491324

Candidate statistics:
Mean: 553.86
Median: 546.0
Minimum: 500
Maximum: 1137

Estimated total candidate rows: 272,123,181
Estimated size: 272.12 million rows


In [ ]:
# Step 51: Define scalable training-data sampling strategy

NEGATIVES_PER_POSITIVE = 20

print("Training candidate sampling strategy configured.")
print("Global candidate pool:", TOP_GLOBAL_CANDIDATES)
print("Negative samples per positive:", NEGATIVES_PER_POSITIVE)

# Estimate training rows for Chunk 1
chunk_positive_count = int(
    chunk_1["in_next_basket"].sum()
)

estimated_chunk_rows = (
    chunk_positive_count
    + chunk_positive_count * NEGATIVES_PER_POSITIVE
)

print("\nChunk 1 positive candidates:", chunk_positive_count)
print(
    "Estimated sampled training rows:",
    f"{estimated_chunk_rows:,}"
)

print(
    "\nEstimated full temporal training size:",
    f"{estimated_chunk_rows * total_targets / len(chunk_1):,.0f}",
    "rows"
)

Training candidate sampling strategy configured.
Global candidate pool: 500
Negative samples per positive: 20

Chunk 1 positive candidates: 32152
Estimated sampled training rows: 675,192

Estimated full temporal training size: 339,076 rows


In [ ]:
# Step 52: Create sampled training chunk

sampled_positive = chunk_1[
    chunk_1["in_next_basket"] == 1
].copy()

sampled_negative = (
    chunk_1[chunk_1["in_next_basket"] == 0]
    .groupby("order_id", group_keys=False)
    .apply(
        lambda x: x.sample(
            n=min(
                len(x),
                len(x) // NEGATIVES_PER_POSITIVE
                if False else
                len(sampled_positive[
                    sampled_positive["order_id"] == x.name
                ]) * NEGATIVES_PER_POSITIVE
            ),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

sampled_chunk_1 = pd.concat(
    [sampled_positive, sampled_negative],
    ignore_index=True
)

sampled_chunk_1 = sampled_chunk_1.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Sampled training chunk created.")
print("Shape:", sampled_chunk_1.shape)

print("\nPositive labels:",
      int(sampled_chunk_1["in_next_basket"].sum()))

print("Negative labels:",
      int((sampled_chunk_1["in_next_basket"] == 0).sum()))

print("\nPositive rate:",
      round(sampled_chunk_1["in_next_basket"].mean() * 100, 2),
      "%")

print("\nUnique orders:",
      sampled_chunk_1["order_id"].nunique())

print("\nSample:")
print(sampled_chunk_1.head(10))

Sampled training chunk created.
Shape: (557109, 5)

Positive labels: 32152
Negative labels: 524957

Positive rate: 5.77 %

Unique orders: 4796

Sample:
   order_id  user_id  order_number  product_id  in_next_basket
0   1866868      812            26       11193               0
1     46069      823             4       22935               0
2    147997     1795            15       27548               0
3    436007      217             8       33000               0
4    401829      264             4       36865               0
5   1561870      432            52       43154               0
6    749498     1540            28       34050               0
7   1437649       71            14       24838               0
8   2012256     1910             4        4799               0
9   2576507      248            31       47209               0


/tmp/ipykernel_1556/753166102.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [ ]:
# Step 53: Validate sampled training chunk

positive_counts = (
    sampled_chunk_1[
        sampled_chunk_1["in_next_basket"] == 1
    ]
    .groupby("order_id")
    .size()
)

negative_counts = (
    sampled_chunk_1[
        sampled_chunk_1["in_next_basket"] == 0
    ]
    .groupby("order_id")
    .size()
)

print("Sampled training validation completed.")

print("\nPositive rows:", len(sampled_positive))
print("Sampled negative rows:", len(sampled_negative))
print("Total rows:", len(sampled_chunk_1))

print("\nOrders with positive candidates:",
      len(positive_counts))

print("Orders with no positive candidates:",
      len(chunk_targets) - len(positive_counts))

print("\nPositive candidates per order:")
print(positive_counts.describe())

print("\nNegative candidates per order:")
print(negative_counts.describe())

# Check that every positive row is truly in the target basket
label_check = True

for row in sampled_positive.itertuples(index=False):
    if row.product_id not in target_product_lookup[row.order_id]:
        label_check = False
        break

print("\nPositive-label check:", label_check)

# Check duplicate candidate rows
duplicate_rows = sampled_chunk_1.duplicated(
    subset=["order_id", "product_id"]
).sum()

print("Duplicate order-product rows:", duplicate_rows)

Sampled training validation completed.

Positive rows: 32152
Sampled negative rows: 524957
Total rows: 557109

Orders with positive candidates: 4796
Orders with no positive candidates: 204

Positive candidates per order:
count    4796.000000
mean        6.703920
std         5.585091
min         1.000000
25%         3.000000
50%         5.000000
75%         9.000000
max        44.000000
dtype: float64

Negative candidates per order:
count    4796.000000
mean      109.457256
std        64.597743
min        20.000000
25%        60.000000
50%       100.000000
75%       160.000000
max       411.000000
dtype: float64

Positive-label check: True
Duplicate order-product rows: 0


In [ ]:
# Step 54: Generate and save all sampled training chunks

import os
import gc

os.makedirs(candidate_output_path, exist_ok=True)

NEGATIVES_PER_POSITIVE = 20
CHUNK_SIZE = 5000

for chunk_number, start_idx in enumerate(
    range(0, len(target_order_lookup), CHUNK_SIZE),
    start=1
):

    end_idx = min(
        start_idx + CHUNK_SIZE,
        len(target_order_lookup)
    )

    chunk_targets = target_order_lookup.iloc[
        start_idx:end_idx
    ]

    rows = []

    for row in chunk_targets.itertuples(index=False):

        order_id = int(row.order_id)

        candidates = combined_candidate_lookup[order_id]
        actual_products = target_product_lookup[order_id]

        positives = [
            int(product_id)
            for product_id in candidates
            if product_id in actual_products
        ]

        if len(positives) == 0:
            continue

        positive_set = set(positives)

        negatives = [
            int(product_id)
            for product_id in candidates
            if product_id not in positive_set
        ]

        sample_size = min(
            len(negatives),
            len(positives) * NEGATIVES_PER_POSITIVE
        )

        rng = np.random.default_rng(
            42 + chunk_number
        )

        if len(negatives) > sample_size:
            selected_negatives = rng.choice(
                negatives,
                size=sample_size,
                replace=False
            )
        else:
            selected_negatives = negatives

        for product_id in positives:
            rows.append({
                "order_id": order_id,
                "user_id": int(row.user_id),
                "order_number": int(row.order_number),
                "product_id": product_id,
                "in_next_basket": 1
            })

        for product_id in selected_negatives:
            rows.append({
                "order_id": order_id,
                "user_id": int(row.user_id),
                "order_number": int(row.order_number),
                "product_id": int(product_id),
                "in_next_basket": 0
            })

    chunk_df = pd.DataFrame(rows)

    output_file = os.path.join(
        candidate_output_path,
        f"candidate_training_chunk_{chunk_number:03d}.parquet"
    )

    chunk_df.to_parquet(
        output_file,
        index=False
    )

    print(
        f"Chunk {chunk_number:03d}/{total_chunks} saved | "
        f"Rows: {len(chunk_df):,} | "
        f"Orders: {chunk_df['order_id'].nunique():,}"
    )

    del rows, chunk_df
    gc.collect()

print("\nAll candidate training chunks generated successfully.")
print("Output directory:", candidate_output_path)

Chunk 001/99 saved | Rows: 735,366 | Orders: 4,855
Chunk 002/99 saved | Rows: 755,187 | Orders: 4,865
Chunk 003/99 saved | Rows: 752,446 | Orders: 4,874
Chunk 004/99 saved | Rows: 738,473 | Orders: 4,879
Chunk 005/99 saved | Rows: 766,903 | Orders: 4,853
Chunk 006/99 saved | Rows: 772,251 | Orders: 4,867
Chunk 007/99 saved | Rows: 777,867 | Orders: 4,884
Chunk 008/99 saved | Rows: 773,071 | Orders: 4,845
Chunk 009/99 saved | Rows: 754,180 | Orders: 4,875
Chunk 010/99 saved | Rows: 740,637 | Orders: 4,843
Chunk 011/99 saved | Rows: 774,745 | Orders: 4,868
Chunk 012/99 saved | Rows: 771,406 | Orders: 4,861
Chunk 013/99 saved | Rows: 766,357 | Orders: 4,880
Chunk 014/99 saved | Rows: 755,458 | Orders: 4,861
Chunk 015/99 saved | Rows: 758,892 | Orders: 4,868
Chunk 016/99 saved | Rows: 777,717 | Orders: 4,847
Chunk 017/99 saved | Rows: 761,908 | Orders: 4,859
Chunk 018/99 saved | Rows: 746,371 | Orders: 4,844
Chunk 019/99 saved | Rows: 746,442 | Orders: 4,875
Chunk 020/99 saved | Rows: 764,

In [ ]:
# Step 55: Verify all saved training chunks

chunk_files = sorted([
    f for f in os.listdir(candidate_output_path)
    if f.startswith("candidate_training_chunk_")
    and f.endswith(".parquet")
])

print("Saved chunk files:", len(chunk_files))

print("\nFirst 3 files:")
print(chunk_files[:3])

print("\nLast 3 files:")
print(chunk_files[-3:])

# Check expected chunk numbering
expected_files = [
    f"candidate_training_chunk_{i:03d}.parquet"
    for i in range(1, total_chunks + 1)
]

missing_files = [
    f for f in expected_files
    if f not in chunk_files
]

print("\nMissing chunk files:", len(missing_files))

if len(missing_files) == 0:
    print("STATUS: All 99 chunks are present.")
else:
    print("Missing files:")
    print(missing_files)

Saved chunk files: 99

First 3 files:
['candidate_training_chunk_001.parquet', 'candidate_training_chunk_002.parquet', 'candidate_training_chunk_003.parquet']

Last 3 files:
['candidate_training_chunk_097.parquet', 'candidate_training_chunk_098.parquet', 'candidate_training_chunk_099.parquet']

Missing chunk files: 0
STATUS: All 99 chunks are present.


In [ ]:
# Step 56: Calculate final training-dataset statistics

total_rows = 0
total_positive = 0
total_negative = 0
total_orders = set()

for file_name in chunk_files:

    file_path = os.path.join(
        candidate_output_path,
        file_name
    )

    df = pd.read_parquet(file_path)

    total_rows += len(df)
    total_positive += int(df["in_next_basket"].sum())
    total_negative += int(
        (df["in_next_basket"] == 0).sum()
    )

    total_orders.update(
        df["order_id"].unique()
    )

    del df

print("Final training-dataset statistics")
print("----------------------------------")
print("Chunk files:", len(chunk_files))
print("Total rows:", f"{total_rows:,}")
print("Positive rows:", f"{total_positive:,}")
print("Negative rows:", f"{total_negative:,}")
print("Unique target orders:", f"{len(total_orders):,}")

print(
    "Positive rate:",
    round(total_positive / total_rows * 100, 2),
    "%"
)

print(
    "Negative-to-positive ratio:",
    round(total_negative / total_positive, 2)
)

print(
    "\nAverage rows per target order:",
    round(total_rows / len(total_orders), 2)
)

Final training-dataset statistics
----------------------------------
Chunk files: 99
Total rows: 74,633,064
Positive rows: 3,579,405
Negative rows: 71,053,659
Unique target orders: 477,785
Positive rate: 4.8 %
Negative-to-positive ratio: 19.85

Average rows per target order: 156.21


In [ ]:
# Step 57: Final integrity validation

duplicate_rows = 0
invalid_labels = 0
orders_checked = 0

for file_name in chunk_files:

    file_path = os.path.join(
        candidate_output_path,
        file_name
    )

    df = pd.read_parquet(file_path)

    duplicate_rows += int(
        df.duplicated(
            subset=["order_id", "product_id"]
        ).sum()
    )

    invalid_labels += int(
        (~df["in_next_basket"].isin([0, 1])).sum()
    )

    orders_checked += df["order_id"].nunique()

    del df

print("FINAL INTEGRITY CHECK")
print("---------------------")
print("Chunks checked:", len(chunk_files))
print("Duplicate order-product rows:", duplicate_rows)
print("Invalid labels:", invalid_labels)
print("Target-order occurrences across chunks:", f"{orders_checked:,}")

if duplicate_rows == 0 and invalid_labels == 0:
    print("\nSTATUS: DATASET INTEGRITY CHECK PASSED")
else:
    print("\nSTATUS: REVIEW REQUIRED")

FINAL INTEGRITY CHECK
---------------------
Chunks checked: 99
Duplicate order-product rows: 0
Invalid labels: 0
Target-order occurrences across chunks: 477,785

STATUS: DATASET INTEGRITY CHECK PASSED
